# Hydrogen Bond Analysis - MD Simulation
## 264THM-PPARG and Luteolin-PDE5A Complexes

This notebook performs hydrogen bond analysis on MD simulation trajectories using GROMACS.

## Step 1: Install GROMACS

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq gromacs > /dev/null 2>&1
gmx --version | head -5

## Step 2: Setup Working Directory and Extract Data

In [ ]:
import os
import subprocess
import zipfile
import numpy as np
import matplotlib.pyplot as plt

# Setup paths
DATASET_PATH = '/kaggle/input/md-hbond-analysis'
WORK_DIR = '/kaggle/working'

# Create output directories
os.makedirs(f'{WORK_DIR}/264THM_PPARG', exist_ok=True)
os.makedirs(f'{WORK_DIR}/Luteolin_PDE5A', exist_ok=True)

# Extract zipped data
for complex_name in ['264THM_PPARG', 'Luteolin_PDE5A']:
    zip_path = f'{DATASET_PATH}/{complex_name}.zip'
    extract_dir = f'{WORK_DIR}/{complex_name}'
    if os.path.exists(zip_path):
        print(f'Extracting {complex_name}...')
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_dir)
        print(f'  Files: {os.listdir(extract_dir)}')
    else:
        print(f'Warning: {zip_path} not found')

print('\nSetup complete!')

## Step 3: Create Index Groups for H-Bond Analysis

In [ ]:
def create_index_file(work_dir, complex_name, ligand_name):
    """Create index file with Protein and Ligand groups"""
    tpr_file = f'{work_dir}/md.tpr'
    ndx_file = f'{work_dir}/hbond.ndx'
    
    # First, create default index
    cmd = f'echo "q" | gmx make_ndx -f {tpr_file} -o {ndx_file}'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    # Check available groups
    print(f'\n=== Index groups for {complex_name} ===')
    cmd = f'gmx make_ndx -f {tpr_file} -n {ndx_file} <<< "q"'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    # Parse output to find Protein and LIG groups
    output = result.stdout + result.stderr
    for line in output.split('\n'):
        if 'Protein' in line or 'LIG' in line or 'Other' in line:
            print(line)
    
    return ndx_file

# Create index for both complexes
for complex_name, ligand in [('264THM_PPARG', 'LIG'), ('Luteolin_PDE5A', 'LIG')]:
    work_dir = f'{WORK_DIR}/{complex_name}'
    create_index_file(work_dir, complex_name, ligand)

## Step 4: Hydrogen Bond Analysis

Using `gmx hbond` to calculate:
- Number of H-bonds between protein and ligand over time
- H-bond lifetime and occupancy

In [ ]:
def run_hbond_analysis(work_dir, complex_name):
    """Run GROMACS hbond analysis"""
    tpr_file = f'{work_dir}/md.tpr'
    xtc_file = f'{work_dir}/trajectory_clean.xtc'
    ndx_file = f'{work_dir}/hbond.ndx'
    
    output_num = f'{work_dir}/hbond_num.xvg'
    output_dist = f'{work_dir}/hbond_dist.xvg'
    output_ang = f'{work_dir}/hbond_ang.xvg'
    output_hbn = f'{work_dir}/hbond.ndx'
    
    print(f'\n=== Running H-bond analysis for {complex_name} ===')
    print(f'TPR: {tpr_file}')
    print(f'XTC: {xtc_file}')
    
    # Run gmx hbond: select Protein (1) and LIG (typically 13 or similar)
    # First find the LIG group number
    cmd = f'echo "q" | gmx make_ndx -f {tpr_file} -o /tmp/temp.ndx 2>&1'
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = result.stdout + result.stderr
    
    # Find LIG group number
    lig_group = None
    protein_group = '1'  # Usually Protein is group 1
    for line in output.split('\n'):
        if 'LIG' in line and ':' in line:
            parts = line.split()
            lig_group = parts[0]
            print(f'Found LIG group: {lig_group}')
            break
    
    if lig_group is None:
        # Try to find 'Other' group which might contain ligand
        for line in output.split('\n'):
            if 'Other' in line and ':' in line:
                parts = line.split()
                lig_group = parts[0]
                print(f'Using Other group as ligand: {lig_group}')
                break
    
    if lig_group:
        # Run H-bond analysis
        cmd = f'echo "{protein_group} {lig_group}" | gmx hbond -f {xtc_file} -s {tpr_file} -num {output_num} -dist {output_dist} -ang {output_ang}'
        print(f'Running: {cmd}')
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        print(result.stdout[-1000:] if len(result.stdout) > 1000 else result.stdout)
        if result.returncode != 0:
            print(f'Error: {result.stderr[-500:]}')
        else:
            print(f'H-bond analysis complete. Output: {output_num}')
    else:
        print('ERROR: Could not find LIG group!')
    
    return output_num, output_dist, output_ang

# Run for both complexes
results = {}
for complex_name in ['264THM_PPARG', 'Luteolin_PDE5A']:
    work_dir = f'{WORK_DIR}/{complex_name}'
    results[complex_name] = run_hbond_analysis(work_dir, complex_name)

## Step 5: Parse and Visualize Results

In [ ]:
def parse_xvg(filename):
    """Parse GROMACS xvg file"""
    time = []
    values = []
    if not os.path.exists(filename):
        print(f'File not found: {filename}')
        return np.array([]), np.array([])
    
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('@'):
                continue
            parts = line.split()
            if len(parts) >= 2:
                time.append(float(parts[0]))
                values.append(float(parts[1]))
    return np.array(time), np.array(values)

# Load and plot H-bond data
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, complex_name in enumerate(['264THM_PPARG', 'Luteolin_PDE5A']):
    work_dir = f'{WORK_DIR}/{complex_name}'
    
    # Number of H-bonds
    hbond_file = f'{work_dir}/hbond_num.xvg'
    time, hbonds = parse_xvg(hbond_file)
    
    if len(time) > 0:
        # Convert to ns
        time_ns = time / 1000
        
        # Plot H-bond number over time
        ax = axes[idx, 0]
        ax.plot(time_ns, hbonds, 'b-', alpha=0.7, linewidth=0.5)
        ax.axhline(np.mean(hbonds), color='r', linestyle='--', label=f'Mean: {np.mean(hbonds):.2f}')
        ax.set_xlabel('Time (ns)')
        ax.set_ylabel('Number of H-bonds')
        ax.set_title(f'{complex_name} - H-bonds over Time')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # H-bond histogram
        ax = axes[idx, 1]
        ax.hist(hbonds, bins=range(int(min(hbonds)), int(max(hbonds))+2), 
                edgecolor='black', alpha=0.7)
        ax.axvline(np.mean(hbonds), color='r', linestyle='--', label=f'Mean: {np.mean(hbonds):.2f}')
        ax.set_xlabel('Number of H-bonds')
        ax.set_ylabel('Frequency')
        ax.set_title(f'{complex_name} - H-bond Distribution')
        ax.legend()
        
        # Print statistics
        print(f'\n=== {complex_name} H-bond Statistics ===')
        print(f'Mean H-bonds: {np.mean(hbonds):.2f} ± {np.std(hbonds):.2f}')
        print(f'Max H-bonds: {np.max(hbonds):.0f}')
        print(f'Min H-bonds: {np.min(hbonds):.0f}')
        print(f'Occupancy (>0 H-bonds): {(hbonds > 0).sum() / len(hbonds) * 100:.1f}%')
    else:
        print(f'No data for {complex_name}')

plt.tight_layout()
plt.savefig(f'{WORK_DIR}/hbond_analysis_combined.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'\nPlot saved to: {WORK_DIR}/hbond_analysis_combined.png')

## Step 6: Summary Statistics

In [ ]:
print('='*60)
print('HYDROGEN BOND ANALYSIS SUMMARY')
print('='*60)

summary_data = []

for complex_name in ['264THM_PPARG', 'Luteolin_PDE5A']:
    work_dir = f'{WORK_DIR}/{complex_name}'
    hbond_file = f'{work_dir}/hbond_num.xvg'
    time, hbonds = parse_xvg(hbond_file)
    
    if len(time) > 0:
        data = {
            'Complex': complex_name,
            'Mean H-bonds': f'{np.mean(hbonds):.2f}',
            'Std': f'{np.std(hbonds):.2f}',
            'Max': f'{np.max(hbonds):.0f}',
            'Min': f'{np.min(hbonds):.0f}',
            'Occupancy (%)': f'{(hbonds > 0).sum() / len(hbonds) * 100:.1f}'
        }
        summary_data.append(data)
        print(f"\n{complex_name}:")
        for k, v in data.items():
            if k != 'Complex':
                print(f"  {k}: {v}")

print('\n' + '='*60)

## Step 7: Save Output Files

In [ ]:
# List all output files
print('Output files:')
for complex_name in ['264THM_PPARG', 'Luteolin_PDE5A']:
    work_dir = f'{WORK_DIR}/{complex_name}'
    print(f'\n{complex_name}:')
    for f in os.listdir(work_dir):
        if f.endswith('.xvg') or f.endswith('.png'):
            filepath = f'{work_dir}/{f}'
            size = os.path.getsize(filepath) / 1024
            print(f'  {f} ({size:.1f} KB)')

print(f'\nCombined plot: {WORK_DIR}/hbond_analysis_combined.png')